# Brain Tumor Segmentation — Kaggle Notebook Training

Backup training environment for when Colab's free GPU quota runs out — Kaggle gives a **separate** free GPU quota (~30 hrs/week, T4 x2 or P100), and the BraTS20 dataset can be attached directly as input with no re-download needed.

**Before running, in the notebook's right sidebar:**
1. **Add Data** -> search `brats20-dataset-training-validation` (by `awsaf49`) -> Add.
2. **Settings -> Accelerator** -> GPU T4 x2 (or P100).
3. **Settings -> Internet** -> On (needed for `git clone` and `pip install`).

## 1. Clone repo and install dependencies

In [ ]:
!git clone https://github.com/thatavidreader/brain_tumor_segmentation.git
%cd brain_tumor_segmentation
!pip install -q -r requirements.txt

## 2. Locate the attached BraTS dataset
Kaggle mounts attached data read-only under `/kaggle/input/`. We search for it rather than hardcoding the path, since the exact nesting can vary.

In [ ]:
import glob
import os

matches = glob.glob("/kaggle/input/**/BraTS20_Training_001", recursive=True)
if not matches:
    raise FileNotFoundError(
        "Could not find BraTS20_Training_001 under /kaggle/input. "
        "Make sure the brats20-dataset-training-validation dataset is attached (Add Data, right sidebar)."
    )
data_dir = os.path.dirname(matches[0])
print("Found dataset at:", data_dir)

## 3. Resume from a previous run (optional)

Kaggle sessions don't share a persistent disk like Drive. To carry checkpoints across sessions:
1. At the end of a session, click **Save Version** (top right) so `/kaggle/working/checkpoints` is saved as this notebook's output.
2. Next session, **Add Data -> Notebook Output Files** -> select your previous version of this same notebook -> Add.
3. Run the cell below — it copies `last_checkpoint.pth` from that attached output into `/kaggle/working/checkpoints` so `train.py` picks it up and resumes automatically.

If this is your first run, skip this cell (no previous checkpoint exists yet).

In [ ]:
import shutil

prev_ckpts = glob.glob("/kaggle/input/**/last_checkpoint.pth", recursive=True)
os.makedirs("/kaggle/working/checkpoints", exist_ok=True)
if prev_ckpts:
    shutil.copy(prev_ckpts[0], "/kaggle/working/checkpoints/last_checkpoint.pth")
    print("Restored checkpoint from:", prev_ckpts[0])
else:
    print("No previous checkpoint found under /kaggle/input — starting fresh.")

## 4. Apply config overrides
Same memory-safe settings worked out on Colab (128^3 patches x batch_size=2 x samples_per_case=4 caused a CUDA OOM on a single T4; cache_rate=0.5 caused a RAM OOM during dataset caching). `checkpoint_dir` points at `/kaggle/working/checkpoints` so `train.py`'s resume logic (`last_checkpoint.pth`) works, and so the checkpoint gets saved when you click Save Version.

In [ ]:
import yaml

with open("config.yaml") as f:
    config = yaml.safe_load(f)

config["paths"]["data_dir"] = data_dir
config["paths"]["checkpoint_dir"] = "/kaggle/working/checkpoints"
config["train"]["epochs"] = 100
config["data"]["cache_rate"] = 0.05
config["train"]["batch_size"] = 1
config["train"]["samples_per_case"] = 2
config["data"]["patch_size"] = [96, 96, 96]

with open("config.yaml", "w") as f:
    yaml.safe_dump(config, f)

print(yaml.safe_dump(config))

## 5. Train
`data/raw` is read-only (mounted from `/kaggle/input`), which is fine — the pipeline only reads volumes from there; `data/splits.json`, logs, and checkpoints all write under the repo's working directory / `/kaggle/working`.

In [ ]:
!python -m src.train --config config.yaml

## 6. Before ending the session
Click **Save Version** (top right, "Save & Run All" or "Quick Save") so `/kaggle/working/checkpoints/last_checkpoint.pth` and `best_model.pth` persist as this notebook's output — otherwise they're wiped when the session ends and step 3 above will have nothing to resume from next time.